# Imports

In [2]:
import pandas as pd
import os
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point

# Merge Roads and Bridges Data

In [7]:
# BASE_DIR = Path(__file__).parent
# DATA_DIR = BASE_DIR / "data"
data_dir = "../data/"

raw_df      = pd.read_csv("../data/_roads3.csv")
bridge_info = pd.read_excel("../data/BMMS_overview.xlsx")
# print(f"  {len(raw_df):,} LRP rows  |  {raw_df['road'].nunique()} roads  |  "
#       f"{len(bridge_info):,} bridge records")

df = raw_df.copy()
df["id"] = range(len(df))

df["length"] = (
    df.groupby("road")["chainage"]
    .diff().abs().fillna(0) * 1000
)

# normalise strings
df["road"] = df["road"].astype(str).str.strip()
df["lrp"]  = df["lrp"].astype(str).str.strip()
df["name"] = df["name"].astype(str).fillna("").str.strip()

# type_simple: Bridge or raw type prefix
df["type_simple"] = df["type"].apply(
    lambda x: "Bridge"
    if isinstance(x, str) and "bridge" in x.lower()
    else str(x).split(" / ")[0].split("%")[0]
)
# merge and aggregate bridge
bridge_info.columns    = bridge_info.columns.str.strip()
bridge_info["LRPName"] = bridge_info["LRPName"].astype(str).str.strip()

def aggregate_bridge_group(group):
    names_str  = " ".join(group["name"].dropna().astype(str).tolist()).upper()
    is_lr_pair = (("(L)" in names_str or " L " in names_str) and
                  ("(R)" in names_str or " R " in names_str))
    return pd.Series({
        "length":    group["length"].median() if is_lr_pair else group["length"].mean(),
        "condition": group["condition"].max(),   # worst condition (D > A)
        "name":      group["name"].iloc[0],
    })

bridge_info_clean = (
    bridge_info.groupby("LRPName", dropna=False)
    .apply(aggregate_bridge_group)
    .reset_index()
    .rename(columns={"length": "length_bmms"})
)

df = df.merge(
    bridge_info_clean,
    left_on="lrp", right_on="LRPName",
    how="left", suffixes=("", "_bmms")
)

mask_bridge       = df["type_simple"] == "Bridge"
mask_has_bmms_len = df["length_bmms"].notna()

df.loc[mask_bridge & mask_has_bmms_len, "length"] = (
    df.loc[mask_bridge & mask_has_bmms_len, "length_bmms"]
)
if "condition_bmms" in df.columns:
    df.loc[mask_bridge, "condition"] = df.loc[mask_bridge, "condition_bmms"]
df.loc[mask_bridge, "condition"] = df.loc[mask_bridge, "condition"].fillna("Unknown")

df = df[df["road"].str.match(r"^[NR]", na=False)].copy()

df = df.sort_values(["road", "chainage"]).reset_index(drop=True)
print(f"  After preprocessing: {len(df):,} rows  |  bridges: {mask_bridge.sum()}")

output_path = "../data/roads_after_dataanalysis.csv"
df.to_csv(output_path, index=False)

  After preprocessing: 19,512 rows  |  bridges: 9101


# Load criticality data

In [ ]:
col_names = [
    "link_no", "name",
    "start_lrp", "start_offset", "start_chainage",
    "end_lrp", "end_offset", "end_chainage",
    "length_km",
    "heavy_truck", "medium_truck", "small_truck",
    "large_bus", "medium_bus", "micro_bus",
    "utility", "car", "auto_rickshaw", "motorcycle", "bicycle",
    "cycle_rickshaw", "cart",
    "motorized_total", "non_motorized_total",
    "total_aadt", "aadt"
]

data_dir = "../data/traffic/"
all_dfs = []

for filename in os.listdir(data_dir):
    if filename.endswith(".traffic.htm"):
        filepath = os.path.join(data_dir, filename)
        try:
            tables = pd.read_html(filepath, encoding="iso-8859-1", match="List of links")
            df = tables[0].copy()
            df.columns = col_names
            # Keep only rows where link_no looks like an actual link, e.g. "N102-1"
            df = df[df["link_no"].str.strip().str.match(r"^[A-Z]+\d+-\d+", na=False)]
            df = df.reset_index(drop=True)
            df["road"] = filename.replace(".traffic.htm", "")
            all_dfs.append(df)
        except Exception as e:
            print(f"Skipped {filename}: {e}")

combined = pd.concat(all_dfs, ignore_index=True)


In [ ]:
combined.tail(20)


# Load vulnerability data

In [ ]:
# Load Flood Data
gdf_flood = gpd.read_file('../data/flood shapefile/bgd_nhr_floods_sparsso.shp')
# Normalise intensity (Severity mapping: 0→0, Severe→1.0, Moderate→0.6, Low→0.3)
flood_intensity = {0:0.0, 1:1.0, 2:0.6, 3:0.3, 4:1.0, 5:0.6, 6:0.3, 7:1.0, 8:0.6}
gdf_flood['flood_hazard_val'] = gdf_flood['FLOODCAT'].astype(int).map(flood_intensity)

# Load Earthquake Data
gdf_eq = gpd.read_file('../data/earthquake shapefile/bgd_nhr_earthquake_sparsso.shp')
eq_hazard_map = {'I': 1.0, 'II': 0.6, 'III': 0.3}
gdf_eq['eq_hazard_val'] = gdf_eq['ZONE'].map(eq_hazard_map).fillna(0.3)

print(f"Flood: {len(gdf_flood)} polygons; EQ: {len(gdf_eq)} polygons.")

In [ ]:
df_all_roads = ("Claude help with a dataframe, suitable to then after use in model.py (potentially with small (structure) changes in model.py")